In [ ]:
import numpy as np
from astropy.table import Table
from astropy.coordinates import SpectralCoord
import astropy.units as u
import matplotlib.pyplot as plt

In [ ]:
# Aux functions
def gaussian_profile(vel_array, intensity, velocity, width):
    sigma = width / (2 * np.sqrt(2 * np.log(2)))
    return intensity * np.exp(-(vel_array - velocity)**2 / (2 * sigma**2))
    # return int * np.exp(-(vel_array - velocity)**2 / (2 * width**2))

In [ ]:
# Read spectrum data
spectrum_filename = "cross-corr-dats/POband7-obsfreq.dat"
spectrum_freq_unit = u.GHz

spectrum_table = Table.read(spectrum_filename, format='ascii')
spec_obsfreq_value = spectrum_table.columns[0].data
spectrum_flux_value = spectrum_table.columns[1].data
spec_obsfreq = spec_obsfreq_value * spectrum_freq_unit

In [ ]:
# Read SLIM data (should be only one)
molecule = "PO,v=0"
slim_freq_unit = u.MHz

slim_filename = "cross-corr-dats/slim_table_region_10.csv"
slim_filename = "cross-corr-dats/SLIM-table-SO.csv"
slim_table = Table.read(slim_filename, format='csv', delimiter=",")
slim_table["Frequency"].unit = slim_freq_unit

In [ ]:
molecule_filter = (slim_table["Formula"] == molecule)

freq_range_filter = ((slim_table["Frequency"] > spec_obsfreq.min()) &
                     (slim_table["Frequency"] < spec_obsfreq.max()))

slim_table_mol = slim_table[molecule_filter & freq_range_filter]
slim_table_mol

In [ ]:
group_col = "Transition" if "Transition" in slim_table_mol.colnames else "Frequency"
slim_table_mol_grouped = slim_table_mol.group_by(group_col)

for key, group in zip(slim_table_mol_grouped.groups.keys,
                      slim_table_mol_grouped.groups):
    # Here group is each transition table of the molecule

    q = group["Frequency"].quantity

    # Check if all rows are the same transition
    if "Transition" in group.colnames:
        if np.all(group["Transition"] == group["Transition"][0]):
            spec_obsfreq_sc = SpectralCoord(
                spec_obsfreq,
                doppler_rest=group["Frequency"].quantity[0],
                doppler_convention='radio',
            )
    elif np.allclose(
        q.to_value(u.MHz), q[1].to_value(u.MHz),
        rtol=1e-12, atol=1e-12):  # This has precision up to the Hz on MHz arrays
        spec_obsfreq_sc = SpectralCoord(
            spec_obsfreq,
            doppler_rest=group["Frequency"].quantity[0],
            doppler_convention='radio',
        )
    else: print("error")

    component_lines = np.array([
        group["Intensity"],
        group["Velocity"],
        group["Width"],
    ]).T
    component_lines

    # Create models
    spec_vel_value = spec_obsfreq_sc.to(u.km / u.s).value
    model = np.zeros_like(spec_vel_value)

    for intensity, vel, width in component_lines:
        model += gaussian_profile(spec_vel_value, intensity, vel, width)

    # Results 

    figsize=(6,2)
    fig, ax = plt.subplots(1, 1, figsize=figsize)

    ax.plot(spec_vel_value, spectrum_flux_value, drawstyle='steps-mid', 
                color="black", alpha=0.8)
    ax.plot(spec_vel_value, model, drawstyle='steps-mid', color="red")
    # ax.axvline(0.0, color="k", ls="--", alpha=0.5)
    ax.set_ylabel("Intensity")
    ax.set_title("SLIM generated model over observed spectrum")
    ax.set_xlabel("Velocity [km/s]")
    ax.text(0.02, 0.98, f"{key[0]}", ha='left', va='top', transform=ax.transAxes)

    ax.set_xlim(-40, 40)
    ax.set_ylim(-8/10*model.max(), 1.5 * model.max())

    plt.tight_layout()
    plt.show()

---

---

---

# correlation part

In [ ]:
# # Compute cross-correlation


# ## Numpy cross-correlation
# corr_np = np.correlate(simulated_data, model, mode='same')

# # Normalize correlation to S/N units
# corr_np /= np.std(corr_np)
# # corr_np /= (np.std(corr_np)**2)

# # Compute lag axis
# lags = (np.arange(n_chan) - n_chan//2) * dv


# ## Scipy FFT cross-correlation

# F_data = np.fft.fft(simulated_data)
# F_model = np.fft.fft(model)
# corr_fft = np.fft.ifft(F_data * np.conj(F_model)).real
# # Move zero lag to the center
# corr_fft = np.fft.fftshift(corr_fft)
# corr_fft /= np.std(corr_fft)

In [ ]:
# # Results 

# fig, axes = plt.subplots(4, 1, figsize=figsize, sharex=True)

# axes[0].plot(vel, model, color="red")
# axes[0].set_ylabel("Model")
# axes[0].set_title("Model molecular emission")

# axes[1].plot(vel, simulated_data, label="Simulated data",
#              color="black", alpha=0.8)
# axes[1].plot(vel, model, color="red", label="True model")
# # axes[1].axvline(0.0, color="k", ls="--", alpha=0.5)
# axes[1].set_ylabel("Intensity")
# axes[1].set_title("Noisy spectrum with true signal location")
# axes[1].set_xlabel("Velocity [km/s]")

# # Old plotting velocity and corr (incorrect, I needed to
# # compute the lag axis)
# # axes[2].plot(vel, corr_np)
# axes[2].plot(lags, corr_np)
# axes[2].set_ylabel("Correlation (S/N)")
# axes[2].set_title("Numpy Cross-correlation")

# axes[3].plot(lags, corr_fft)
# axes[3].set_xlabel("Velocity shift (km/s)")
# axes[3].set_ylabel("Correlation")
# axes[3].set_title("Scipy FFT Cross-correlation")

# peak_sn = np.max(corr_np)
# peak_shift = lags[np.argmax(corr_np)]
# peak_vel = peak_shift + first_vel

# print(f"Peak correlation S/N            = {peak_sn:.2f}")
# print(f"Recovered shift                 = {peak_shift:.3f} km/s")
# print(f"Detected velocity of first comp = {peak_vel:.3f} km/s")
# print(f"True velocity of signal         = {true_vel:.3f} km/s")

# plt.tight_layout()
# plt.show()